# Lesson 5: Why Is a Posterior Hard to Compute? From Monte Carlo to MCMC
### Student workbook
*Course: Bayesian Analysis of Empirical Data (2026)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/05_monte_carlo_to_mcmc.ipynb)

Russian version: [05_monte_carlo_to_mcmc_ru.ipynb](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/05_monte_carlo_to_mcmc_ru.ipynb)


## 1. Before class
* **One-minute goal:** a posterior is "prior × likelihood divided by an area". The area is easy with one unknown and impossible with many. Random draws, and in particular a Markov chain walker, give us the posterior without ever computing that area.
* **Example:** the Lesson 4 housing-subsidy survey: 58 of 100 lower-income households support the subsidy; prior Beta(2, 2); exact posterior Beta(60, 44).
* **How to work:** in every task, **predict first**, then change **one number** and rerun.
* **Tools:** NumPy, SciPy and Matplotlib only. PyMC comes in Lesson 6.


In [ ]:
import sys
import math
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

IS_COLAB = "google.colab" in sys.modules
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})

# The running example (Lesson 4)
SUPPORTERS, HOUSEHOLDS = 58, 100
PRIOR_A, PRIOR_B = 2, 2
EXACT = stats.beta(PRIOR_A + SUPPORTERS, PRIOR_B + HOUSEHOLDS - SUPPORTERS)   # Beta(60, 44)

LBL = {"support": "support among lower-income households", "density": "density",
       "step": "step", "share": "share of nights", "island": "island", "exact": "exact posterior"}
print("Exact P(support > 50%):", round(EXACT.sf(0.5), 3))


## 2. Predict before you run (5 minutes, on paper)
1. With **20 unknowns** and 100 grid points per unknown, how many grid cells are needed?
2. You throw 1,000 random darts at a square containing a quarter circle. Roughly what share lands inside?
3. A walker always moves uphill and sometimes downhill. Where will it spend most of its time?


## 3. Classwork 1 — The area problem
Bayes' theorem gives the posterior **shape**: prior × likelihood. To turn the shape into a distribution we divide by its total **area** (the evidence). With one unknown, a grid of bars computes it.

**Change** `N_BARS` (5, 10, 20, 100) and compare with the exact area.


In [ ]:
def shape(theta):
    """Prior x likelihood: the posterior shape WITHOUT the denominator."""
    return stats.beta.pdf(theta, PRIOR_A, PRIOR_B) * stats.binom.pmf(SUPPORTERS, HOUSEHOLDS, theta)

N_BARS = 10                                   # <- change me
edges = np.linspace(0, 1, N_BARS + 1)
mids = (edges[:-1] + edges[1:]) / 2
area_grid = (shape(mids) * (1 / N_BARS)).sum()
area_exact = stats.betabinom.pmf(SUPPORTERS, HOUSEHOLDS, PRIOR_A, PRIOR_B)
print(f"{N_BARS} bars: area = {area_grid:.5f}   exact area = {area_exact:.5f}")

xs = np.linspace(0, 1, 500)
plt.bar(mids, shape(mids), width=1 / N_BARS * 0.95, alpha=0.4)
plt.plot(xs, shape(xs), lw=2)
plt.xlabel(LBL["support"]); plt.ylabel("prior × likelihood"); plt.show()


Now count the cost of a grid when the model has more unknowns. **Change** `POINTS_PER_UNKNOWN` and the list of unknowns.


In [ ]:
POINTS_PER_UNKNOWN = 100                    # <- change me
CELLS_PER_SECOND = 1e9                       # a fast computer
SECONDS_PER_YEAR = 3.156e7

for d in [1, 2, 3, 5, 10, 20]:
    cells = POINTS_PER_UNKNOWN ** d
    years = cells / CELLS_PER_SECOND / SECONDS_PER_YEAR
    print(f"{d:>2} unknowns: {cells:.1e} cells, {years:.1e} years")


**Question 1.** How many unknowns can a grid handle in about one minute? What does this mean for a model with one parameter per participant?


## 4. Classwork 2 — Darts compute areas (Monte Carlo)
**Part A.** Throw darts at the unit square. The share inside the quarter circle estimates its area, π/4. **Change** `N_DARTS` (10, 100, 1,000, 100,000) and run the cell several times. How much does the estimate wobble?


In [ ]:
N_DARTS = 1000                               # <- change me
x, y = rng.uniform(0, 1, N_DARTS), rng.uniform(0, 1, N_DARTS)
inside = x**2 + y**2 <= 1
print(f"{N_DARTS} darts: pi is about {4 * inside.mean():.4f}  (true value {math.pi:.4f})")


**Part B — rejection sampling (von Neumann, 1951).** Throw darts at a box around the posterior *shape* and keep those under the curve. The kept positions are draws from the posterior. We never use the denominator.


In [ ]:
N_DARTS = 20_000                             # <- change me
box_height = shape(np.linspace(0, 1, 2001)).max() * 1.01
x = rng.uniform(0, 1, N_DARTS)
u = rng.uniform(0, box_height, N_DARTS)
kept = x[u < shape(x)]

print(f"kept {len(kept)} of {N_DARTS} darts ({len(kept) / N_DARTS:.1%})")
print(f"P(support > 50%): darts {np.mean(kept > 0.5):.3f}   exact {EXACT.sf(0.5):.3f}")
print(f"95% interval: darts {np.quantile(kept, [0.025, 0.975]).round(3)}   exact {EXACT.ppf([0.025, 0.975]).round(3)}")

plt.hist(kept, bins=40, density=True, alpha=0.5, label="kept darts")
plt.plot(xs, EXACT.pdf(xs), "k--", label=LBL["exact"])
plt.xlim(0.3, 0.85); plt.xlabel(LBL["support"]); plt.legend(); plt.show()


**Question 2.** What share of darts is wasted? Using Section 5 of the lesson, what would happen with 20 unknowns?


## 5. Classwork 3 — King Markov's islands
Seven islands in a ring with populations 1, 2, …, 7 (thousand). Each night the king proposes a neighbour at random. If it is bigger he moves. If it is smaller he moves with probability = (its population) ÷ (current population).

**Change** `START_ISLAND` (1–7) and `NIGHTS` (10, 100, 1,000, 50,000).


In [ ]:
POPULATION = np.arange(1, 8)
START_ISLAND = 4                             # <- change me (1-7)
NIGHTS = 1000                                # <- change me

def king_markov(start, nights, rng):
    pos = start - 1
    visits = np.zeros(7)
    for _ in range(nights):
        visits[pos] += 1
        proposal = (pos + rng.choice([-1, 1])) % 7
        if rng.random() < POPULATION[proposal] / POPULATION[pos]:   # bigger: ratio > 1, always move
            pos = proposal
    return visits / nights

shares = king_markov(START_ISLAND, NIGHTS, rng)
target = POPULATION / POPULATION.sum()
plt.bar(POPULATION, shares, alpha=0.6, label="nights spent")
plt.scatter(POPULATION, target, color="darkorange", marker="_", s=900, lw=3, label="population share")
plt.xlabel(LBL["island"]); plt.ylabel(LBL["share"]); plt.legend(); plt.show()
print("share on island 7:", round(shares[6], 3), " target:", round(target[6], 3))


**Question 3.** Does the starting island matter after 50,000 nights? After 10 nights? The king never knew the total population of all islands. Which number in the rule plays the role of the "unknown area"?


## 6. Classwork 4 — The Metropolis walker on the survey hill
The same rule on a continuous hill. The walker uses only the **logarithm of prior × likelihood**: logarithms avoid tiny numbers, and a ratio of heights becomes a difference of logarithms.

**Change** `STEP` (0.005, 0.05, 0.5) and `START` (0.05, 0.4, 0.9). The first `WARMUP` steps are discarded (more on this in Lesson 6).


In [ ]:
def log_shape(theta):
    if not 0 < theta < 1:
        return -np.inf                       # impossible values have zero height
    return stats.beta.logpdf(theta, PRIOR_A, PRIOR_B) + stats.binom.logpmf(SUPPORTERS, HOUSEHOLDS, theta)

def metropolis(log_height, start, step, n_steps, rng):
    chain = np.empty(n_steps)
    current, current_lh = start, log_height(start)
    accepted = 0
    for i in range(n_steps):
        proposal = current + rng.normal(0, step)                       # 1. propose a nearby step
        proposal_lh = log_height(proposal)
        if np.log(rng.random()) < proposal_lh - current_lh:            # 2-3. uphill: always; downhill: sometimes
            current, current_lh = proposal, proposal_lh
            accepted += 1
        chain[i] = current                                             # 4. if rejected: stay and count again
    return chain, accepted / n_steps

STEP, START, N_STEPS, WARMUP = 0.05, 0.4, 5000, 500                    # <- change STEP and START
chain, acc = metropolis(log_shape, START, STEP, N_STEPS, rng)
draws = chain[WARMUP:]
print(f"accepted {acc:.0%} of proposals")
print(f"P(support > 50%): walker {np.mean(draws > 0.5):.3f}   exact {EXACT.sf(0.5):.3f}")
print(f"mean: walker {draws.mean():.3f}   exact {EXACT.mean():.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(chain, lw=0.7); ax[0].axvspan(0, WARMUP, color="red", alpha=0.1)
ax[0].set_xlabel(LBL["step"]); ax[0].set_ylabel(LBL["support"][:7])
ax[1].hist(draws, bins=40, density=True, alpha=0.5); ax[1].plot(xs, EXACT.pdf(xs), "k--")
ax[1].set_xlim(0.3, 0.85); ax[1].set_xlabel(LBL["support"]); plt.tight_layout(); plt.show()


**The denominator really is never needed.** Multiply the shape by 1,000 (add log 1000 to its logarithm) and rerun the walker with the same random seed. Compare the two chains.


In [ ]:
chain_a, _ = metropolis(log_shape, 0.4, 0.05, 3000, np.random.default_rng(1))
chain_b, _ = metropolis(lambda t: log_shape(t) + np.log(1000), 0.4, 0.05, 3000, np.random.default_rng(1))
print("Identical chains:", np.allclose(chain_a, chain_b))


**Question 4.** With `STEP = 0.005`, what goes wrong? With `STEP = 0.5`? With `START = 0.05` and `WARMUP = 0`? Write one sentence for each. These are the problems Lesson 6 teaches you to detect.


## 7. 🚀 Stretch — two unknowns at once
Walk over **both** groups together: lower-income (58 of 100) and higher-income (40 of 100). The walker proposes a step in both directions at once. Estimate P(gap > 5 points) and compare with Lesson 4 (exact: 0.964).


In [ ]:
def log_shape_2(t):
    lo, hi = t
    if not (0 < lo < 1 and 0 < hi < 1):
        return -np.inf
    return (stats.beta.logpdf(lo, 2, 2) + stats.binom.logpmf(58, 100, lo)
            + stats.beta.logpdf(hi, 2, 2) + stats.binom.logpmf(40, 100, hi))

current = np.array([0.5, 0.5]); current_lh = log_shape_2(current)
chain2 = np.empty((20_000, 2))
for i in range(len(chain2)):
    proposal = current + rng.normal(0, 0.05, size=2)
    proposal_lh = log_shape_2(proposal)
    if np.log(rng.random()) < proposal_lh - current_lh:
        current, current_lh = proposal, proposal_lh
    chain2[i] = current
draws2 = chain2[2000:]
gap = draws2[:, 0] - draws2[:, 1]
print("P(gap > 5 points):", round(np.mean(gap > 0.05), 3), "  exact: 0.964")
plt.plot(draws2[::10, 0], draws2[::10, 1], ".", ms=2, alpha=0.4)
plt.xlabel("lower-income support"); plt.ylabel("higher-income support"); plt.show()


## 8. 🚀 Stretch — why blind darts fail in many dimensions
Throw random points into a d-dimensional box and count how many land inside the ball, i.e. in the region where the probability is. **Change** the list of dimensions.


In [ ]:
for d in [2, 3, 5, 10, 15]:
    pts = rng.uniform(-1, 1, size=(200_000, d))
    print(f"{d:>2} unknowns: {np.mean((pts**2).sum(axis=1) <= 1):.5%} of points are useful")


## 9. Exit ticket
Answer in two or three sentences each:
1. Why does computing a posterior mean computing an area, and why does a grid fail for many unknowns?
2. Why does the Metropolis walker never need the denominator?
3. A rejected proposal makes the walker stay. Why must the repeated value be kept?

The lesson page with all interactive figures: [Lesson 5](https://iknyazeva.github.io/bayes-cogsci-book/en/montecarlotomcmc/)

---
*Reproducibility:* NumPy random generator with `RANDOM_SEED = 42`. Results change slightly if you rerun cells in a different order; that wobble is the Monte Carlo error.
